# 2 - Text Preprocessing

**Preparing reviews for vectorization, in multiple parallel variants (without normalization, stemmed, lemmatized, and Lancaster), which we further share across all models (notebooks `4.x`-`9.x`).**

Order of text cleaning steps:

1. Preprocessing initialization (calling the `init_preprocessing` function from `src/preprocessing.py`):
    1. Downloading NLTK (Natural Language Toolkit)
    2. Creating a custom stop word list

2. Loading raw data

3. Removing duplicates

4. Cleaning the entire dataset with `basic_clean` (calling the `basic_clean` function from `src/preprocessing.py`), which includes:

    1. Removing HTML (must be done first, because the HTML-encoded apostrophe `&#39;` must become a real apostrophe before we recognize contractions, and HTML tags attached to words would break tokenization if we left them for later)

    2. Lowercase (must be done before expanding contractions, because the contraction dictionary is written in lowercase; otherwise, contractions at the beginning of a sentence would be missed)

    3. Expanding contractions (done before removing punctuation (`don't` → `do not`, including forms without apostrophes such as `isnt` → `is not`, as well as some informal forms such as `gonna` → `going to`))

    4. Removing numbers and punctuation

    5. Tokenization (now we split the cleaned text by spaces)

    6. Removing stop words (a custom list that preserves negations; must be done after tokenization because it operates on individual cleaned words)

    7. Removing tokens shorter than 2 characters

5. Train/test split (80/20 stratified split, performed once, before branching)

6. Branching into multiple parallel variants:
    - Baseline - without normalization (calling the `baseline_tokens` function from `src/preprocessing.py`)
    - Stemming – Snowball stemmer (calling the `stem_tokens` function from `src/preprocessing.py`)
    - Lemmatization – WordNet lemmatizer with POS tagging (calling the `lemmatize_tokens` function from `src/preprocessing.py`)
    - Lancaster - Lancaster stemmer (calling the `lancaster_stem_tokens` function from `src/preprocessing.py`)

7. Checking text cleaning on the same example

8. Saving processed data

<br>

Note: The text cleaning functions (`basic_clean`, which includes: `expand_contractions`, `clean_html`, `remove_non_alpha`), initialization (`init_preprocessing`), and all 4 branches (`baseline_tokens`, `stem_tokens`, `lemmatize_tokens`, `lancaster_stem_tokens`) are defined in `src/preprocessing.py`, instead of being defined here. The same functions are also used by `predict_review.py` (the GUI application for prediction on an entered review) - so that predictions in the GUI are consistent with what the models actually learned, both sides must use identical preprocessing logic, not two separate copies of the same code.


In [1]:
import sys
sys.path.append("..")

import time
import pandas as pd

from pathlib import Path
from sklearn.model_selection import train_test_split

from src.preprocessing import (
    init_preprocessing,
    basic_clean,
    baseline_tokens,
    stem_tokens,
    lemmatize_tokens,
    lancaster_stem_tokens
)

## Preprocessing Initialization

1. Downloading NLTK (Natural Language Toolkit)
    - All NLTK resources required for further text processing are downloaded:
        - Stop words
        - WordNet for lemmatization
        - Models for POS tagging (the name differs between library versions, so both names are listed)

2. Creating a custom stop word list (a custom list that preserves negations)
    - The standard NLTK English list, with words that carry negation removed
    - Negation words are intentionally kept in the text because they carry a significant sentiment signal (e.g. "not good" and "good" are completely different)


In [2]:
init_preprocessing()

stopwords: OK
wordnet: OK
omw-1.4: OK
averaged_perceptron_tagger: OK
averaged_perceptron_tagger_eng: OK
NLTK resources downloaded.

Custom stopword list built.
Number of stopwords in the custom list: 195


## Loading the Data

We check how many rows were loaded.


In [3]:
DATA_PATH = "../data/raw/IMDB_Dataset.csv"

df = pd.read_csv(DATA_PATH)

print(f"Number of rows: {df.shape[0]}")

Number of rows: 50000


## Removing Duplicates

We do this before any text cleaning.

Duplicates are removed from the original, raw data.


In [4]:
rows_before = df.shape[0]
df = df.drop_duplicates().reset_index(drop=True)
rows_after = df.shape[0]

print(f"Duplicates removed: {rows_before - rows_after}")
print(f"Remaining rows: {rows_after}")

Duplicates removed: 418
Remaining rows: 49582


## Cleaning the Entire Dataset (Applying the `basic_clean` Function)

1. Decoding HTML entities (`&#39;` → `'`) as well as removing HTML tags (`<br>`)
2. Converting to lowercase (`.lower()` function)
3. Expanding contractions (`don't` → `do not`)
    - Irregular contractions that cannot be generated by a general rule ("n't → not")
    - Non-negating contractions that we also expand for consistency ("it's → it is")
    - Contractions written without apostrophes (a common informal variant)
    - Several informal expressions common in more casually written reviews ("gonna", "kinda"...)
    - Regular contractions
4. Removing numbers and punctuation and unnecessary spaces (only characters and spaces remain)
5. Tokenization (splitting by spaces)
6. Removing stop words (custom list that preserves negations)
7. Removing tokens shorter than 2 characters (remaining cleaning)

This step cleans all 49,582 reviews.


In [5]:
start = time.time()
df["tokens"] = df["review"].apply(basic_clean)
elapsed = time.time() - start

print(f"Completed in {elapsed:.1f} seconds")
print(f"Average number of tokens per review after cleaning: {df['tokens'].apply(len).mean():.1f}")

Completed in 24.9 seconds
Average number of tokens per review after cleaning: 121.2


## Train/test split

We perform the split only once, here.

We do it before branching into baseline/stemming/lemmatization/Lancaster - this way, all variants share the same rows in the train/test set.

If we split separately for each branch, the comparison of variants in `10_evaluation.ipynb` would be unreliable, because the difference in results would partially come from which reviews ended up in the test set, rather than exclusively from the way the text was processed.

We use `stratify=df["sentiment"]` to ensure the same class ratio in the train and test sets as in the entire dataset, and a fixed `random_state=1` for reproducibility.


In [6]:
train_df, test_df = train_test_split(df, test_size=0.2, stratify=df["sentiment"], random_state=1,)

train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

# Ispis statistike o broju redova i procentu pozitivnih i negativnih primera u train i test skupu
train_stats = pd.DataFrame({
    "Number of rows": [len(train_df)],
    "Positive": [train_df["sentiment"].value_counts(normalize=True).get("positive", 0) * 100],
    "Negative": [train_df["sentiment"].value_counts(normalize=True).get("negative", 0) * 100]
}, index=["Train"])
test_stats = pd.DataFrame({
    "Number of rows": [len(test_df)],
    "Positive": [test_df["sentiment"].value_counts(normalize=True).get("positive", 0) * 100],
    "Negative": [test_df["sentiment"].value_counts(normalize=True).get("negative", 0) * 100]
}, index=["Test"])
stats = pd.concat([train_stats, test_stats])
stats["Positive"] = stats["Positive"].map(lambda x: f"{x:.2f}%")
stats["Negative"] = stats["Negative"].map(lambda x: f"{x:.2f}%")

print(stats)

       Number of rows Positive Negative
Train           39665   50.19%   49.81%
Test             9917   50.19%   49.81%


## Branch A: Baseline (without normalization)

Control variant - uses the same cleaned and tokenized words as the following branches (after `basic_clean`: HTML, lowercase, contractions, numbers/punctuation, stop words, short tokens), but without stemming or lemmatization.

The goal of this branch is to measure whether normalization (reducing words to their stem/lemma) contributes to model performance at all, or whether the models perform equally well on raw (but cleaned) tokens.

Without this branch, we would not have a control point for comparison - we would only know whether stemming is better than lemmatization, not whether either of them is better than the absence of normalization.

Safe to apply to both the train and test sets (not data leakage), since it is a deterministic transformation with fixed rules that does not "learn" anything from our dataset, unlike the TF-IDF from the following notebook, which calculates (fits) IDF weights based on the training set.


In [7]:
start = time.time()
train_df["review_baseline"] = train_df["tokens"].apply(baseline_tokens)
test_df["review_baseline"] = test_df["tokens"].apply(baseline_tokens)
elapsed = time.time() - start

print(f"Completed in {elapsed:.1f} seconds")
print("Example:")
print(train_df["review_baseline"].iloc[0][:300])

Completed in 0.4 seconds
Example:
let start saying love japanese cinema literature culture generally seen many japanese movies enjoyed portrait hell aka jigokuhen makes ridiculous two characters dominate action evil lord privileged bubble stubborn crazy artist pure types zero subtlety nuance actions emanate cartoonish extremes film 


## Branch B: Stemming (Snowball / Porter2)

Stemming is the process of roughly reducing words to their root by removing suffixes, based on predefined rules, without understanding grammar or context.

The goal is to reduce different forms of the same word (e.g. "boring", "bored", "bore") to the same form, thereby reducing the vocabulary size and making it easier for the model to recognize that these words carry similar meaning.

The result is often not a real English word (e.g. "movie" → "movi"), which is expected and does not represent a problem for the model.

Safe to apply to both the train and test sets (not data leakage), since it is a deterministic transformation with fixed rules that does not "learn" anything from our dataset, unlike the TF-IDF from the following notebook, which calculates (fits) IDF weights based on the training set.


In [8]:
start = time.time()
train_df["review_stemmed"] = train_df["tokens"].apply(stem_tokens)
test_df["review_stemmed"] = test_df["tokens"].apply(stem_tokens)
elapsed = time.time() - start

print(f"Stemming completed in {elapsed:.1f} seconds")
print("\nExample:")
print(train_df["review_stemmed"].iloc[0][:300])

Stemming completed in 79.3 seconds

Example:
let start say love japanes cinema literatur cultur general seen mani japanes movi enjoy portrait hell aka jigokuhen make ridicul two charact domin action evil lord privileg bubbl stubborn crazi artist pure type zero subtleti nuanc action eman cartoonish extrem film want show horribl scene violenc ra


## Branch C: Lemmatization (WordNet + POS tagging)

Lemmatization reduces a word to its grammatically correct basic (dictionary) form — the lemma.

It works using a lexical database (WordNet) and information about the part of speech (noun, verb, adjective, adverb).

Lemmatization is slower but more precise: for example, "running" correctly becomes "run", while with coarser methods it often remains unchanged.

The result is always a real English word.

We use POS tagging before lemmatization - without it, `WordNetLemmatizer` would assume that every word is a noun (e.g. "running" would not be reduced to "run").

Safe to apply to both the train and test sets (not data leakage), since it is a deterministic transformation with fixed rules that does not "learn" anything from our dataset, unlike the TF-IDF from the following notebook, which calculates (fits) IDF weights based on the training set.


In [9]:
start = time.time()
train_df["review_lemmatized"] = train_df["tokens"].apply(lemmatize_tokens)
test_df["review_lemmatized"] = test_df["tokens"].apply(lemmatize_tokens)
elapsed = time.time() - start

print(f"Lemmatization completed in {elapsed:.1f} seconds")
print("\nExample:")
print(train_df["review_lemmatized"].iloc[0][:300])

Lemmatization completed in 323.9 seconds

Example:
let start say love japanese cinema literature culture generally see many japanese movie enjoy portrait hell aka jigokuhen make ridiculous two character dominate action evil lord privilege bubble stubborn crazy artist pure type zero subtlety nuance action emanate cartoonish extreme film want show hor


## Branch D: Aggressive Stemming (Lancaster)

The Lancaster stemmer is known to be significantly more aggressive than Snowball (Branch B).

It uses a smaller and coarser set of rules, often producing shorter stems and merging more different words into the same form (e.g. words that Snowball keeps separate may be reduced to the same stem by Lancaster).

Motivation for this branch:
- Since we observed that (moderate) Snowball stemming gives slightly better results than lemmatization, we test whether this trend of "less grammatical precision → better result" is also consistent with more aggressive stemming
- Or whether there is a point beyond which too much aggressiveness starts to hurt (different terms with different meanings lose their distinction)

Safe to apply to both the train and test sets (not data leakage), since it is a deterministic transformation with fixed rules that does not "learn" anything from our dataset, unlike the TF-IDF from the following notebook, which calculates (fits) IDF weights based on the training set.


In [10]:
start = time.time()
train_df["review_lancaster"] = train_df["tokens"].apply(lancaster_stem_tokens)
test_df["review_lancaster"] = test_df["tokens"].apply(lancaster_stem_tokens)
elapsed = time.time() - start

print(f"Lancaster stemming completed in {elapsed:.1f} seconds")
print("\nExample:")
print(train_df["review_lancaster"].iloc[0][:300])

Lancaster stemming completed in 85.5 seconds

Example:
let start say lov japanes cinem lit cult gen seen many japanes movy enjoy portrait hel ak jigokuh mak ridic two charact domin act evil lord privileg bubbl stubborn crazy art pur typ zero subtl nuant act em cartoon extrem film want show horr scen viol raw emot many scen top act becom laugh overal fee


## Checking Text Cleaning on the Same Example

Checking the correctness of the preprocessing functions on an example that includes various cases and elements that need to be removed and processed.


In [11]:
sample = "<p>I thought this movie was gonna be awful, but it wasn't</p>. I didn't wanna expect much, yet it's surprisingly good. <br /><br />There's great acting, and the story's kinda clever."
tokens = basic_clean(sample)
print("Sample:    ", sample)
print("Tokens:    ", tokens)
print("Baseline:  ", " ".join(tokens))
print("Stemmed:   ", stem_tokens(tokens))
print("Lemmatized:", lemmatize_tokens(tokens))
print("Lancaster: ", lancaster_stem_tokens(tokens))

Sample:     <p>I thought this movie was gonna be awful, but it wasn't</p>. I didn't wanna expect much, yet it's surprisingly good. <br /><br />There's great acting, and the story's kinda clever.
Tokens:     ['thought', 'movie', 'going', 'awful', 'not', 'not', 'want', 'expect', 'much', 'yet', 'surprisingly', 'good', 'great', 'acting', 'story', 'kind', 'clever']
Baseline:   thought movie going awful not not want expect much yet surprisingly good great acting story kind clever
Stemmed:    thought movi go aw not not want expect much yet surpris good great act stori kind clever
Lemmatized: thought movie go awful not not want expect much yet surprisingly good great act story kind clever
Lancaster:  thought movy going aw not not want expect much yet surpr good gre act story kind clev


## Saving the Results

We save multiple separate pairs of files (train/test) - one pair for each variant.

Each contains only the two columns needed for further work: processed text and sentiment label.


In [12]:
BASELINE_DIR = Path("../data/processed/baseline")
STEMMED_DIR = Path("../data/processed/stemmed")
LEMMATIZED_DIR = Path("../data/processed/lemmatized")
LANCASTER_DIR = Path("../data/processed/lancaster")

BASELINE_DIR.mkdir(parents=True, exist_ok=True)
STEMMED_DIR.mkdir(parents=True, exist_ok=True)
LEMMATIZED_DIR.mkdir(parents=True, exist_ok=True)
LANCASTER_DIR.mkdir(parents=True, exist_ok=True)

train_df[["review_baseline", "sentiment"]].to_csv(BASELINE_DIR / "train.csv", index=False)
test_df[["review_baseline", "sentiment"]].to_csv(BASELINE_DIR / "test.csv", index=False)

train_df[["review_stemmed", "sentiment"]].to_csv(STEMMED_DIR / "train.csv", index=False)
test_df[["review_stemmed", "sentiment"]].to_csv(STEMMED_DIR / "test.csv", index=False)

train_df[["review_lemmatized", "sentiment"]].to_csv(LEMMATIZED_DIR / "train.csv", index=False)
test_df[["review_lemmatized", "sentiment"]].to_csv(LEMMATIZED_DIR / "test.csv", index=False)

train_df[["review_lancaster", "sentiment"]].to_csv(LANCASTER_DIR / "train.csv", index=False)
test_df[["review_lancaster", "sentiment"]].to_csv(LANCASTER_DIR / "test.csv", index=False)

print("Saved:")
print(f"  {BASELINE_DIR / 'train.csv'} - {train_df.shape[0]} rows")
print(f"  {BASELINE_DIR / 'test.csv'} - {test_df.shape[0]} rows")
print(f"  {STEMMED_DIR / 'train.csv'} - {train_df.shape[0]} rows")
print(f"  {STEMMED_DIR / 'test.csv'} - {test_df.shape[0]} rows")
print(f"  {LEMMATIZED_DIR / 'train.csv'} - {train_df.shape[0]} rows")
print(f"  {LEMMATIZED_DIR / 'test.csv'} - {test_df.shape[0]} rows")
print(f"  {LANCASTER_DIR / 'train.csv'} - {train_df.shape[0]} rows")
print(f"  {LANCASTER_DIR / 'test.csv'} - {test_df.shape[0]} rows")

Saved:
  ..\data\processed\baseline\train.csv - 39665 rows
  ..\data\processed\baseline\test.csv - 9917 rows
  ..\data\processed\stemmed\train.csv - 39665 rows
  ..\data\processed\stemmed\test.csv - 9917 rows
  ..\data\processed\lemmatized\train.csv - 39665 rows
  ..\data\processed\lemmatized\test.csv - 9917 rows
  ..\data\processed\lancaster\train.csv - 39665 rows
  ..\data\processed\lancaster\test.csv - 9917 rows


## Preprocessing Conclusions

- 418 duplicate rows were removed, leaving 49,582 reviews.

- After cleaning, the average review contains 121.2 tokens.

- The custom stop list contains 195 words (the standard NLTK list minus negations).

- Training set: 39,665 reviews, test set: 9,917 reviews, with class balance preserved through stratification.

- Ratio of positive / negative reviews in the Train and Test sets: 50.19% / 49.81%

- Results saved in `data/processed/...`, ready for vectorization in `3_vectorization.ipynb`.
